# Image Colorization with GAN (Pix2Pix) — Local Training

**Architecture:**
- **Generator:** U-Net with Skip-Connections
- **Discriminator:** PatchGAN
- **Color space:** LAB (L = lightness, AB = color)

## Cell 1 – Setup & Imports

In [ ]:
# Install dependencies if needed
# !pip install tensorflow scikit-image matplotlib numpy

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from skimage import color
import os

print("TensorFlow Version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

tf.random.set_seed(42)
np.random.seed(42)

## Cell 2 – Local Paths

Set `IMAGE_DIR` to the folder containing your training images (`.jpg` files).

In [ ]:
# ── Adjust paths ─────────────────────────────────────────────
BASE_DIR        = os.path.abspath(".")          # current notebook directory
IMAGE_DIR       = os.path.join(BASE_DIR, "data", "images")   # put your .jpg images here
CHECKPOINT_DIR  = os.path.join(BASE_DIR, "checkpoints")
PREVIEW_DIR     = os.path.join(BASE_DIR, "previews")
# ─────────────────────────────────────────────────────────────

os.makedirs(IMAGE_DIR,      exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PREVIEW_DIR,    exist_ok=True)

print(f"Image folder:  {IMAGE_DIR}")
print(f"Checkpoints:   {CHECKPOINT_DIR}")
print(f"Previews:      {PREVIEW_DIR}")

## Cell 2b – (Optional) Unzip Dataset

In [ ]:
import zipfile

# Set to None to skip unzipping
ZIP_PATH = None  # e.g. os.path.join(BASE_DIR, "data", "train2017.zip")

if ZIP_PATH and os.path.exists(ZIP_PATH):
    existing = len([f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")])
    if existing == 0:
        print("Extracting dataset...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(IMAGE_DIR)
        print(f"Done: {len(os.listdir(IMAGE_DIR))} files extracted")
    else:
        print(f"Already extracted ({existing} images found)")
else:
    num = len([f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")]) if os.path.exists(IMAGE_DIR) else 0
    print(f"Using existing images in IMAGE_DIR ({num} .jpg files found)")

## Cell 3 – Hyperparameters

In [ ]:
IMG_SIZE    = 256    # image size (256x256)
BATCH_SIZE  = 8     # reduce to 4 if you run out of memory
EPOCHS      = 100   # number of training epochs
LAMBDA      = 100   # L1 loss weight vs. GAN loss
LR          = 2e-4  # learning rate
BETA_1      = 0.5   # Adam beta1
SAVE_EVERY  = 5     # save checkpoint every N epochs

print("Hyperparameters set")

## Cell 4 – Data Pipeline

In [ ]:
def load_and_preprocess(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0

    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)

    img_np = img.numpy()
    lab    = color.rgb2lab(img_np).astype(np.float32)

    L  = lab[:, :, 0:1] / 50.0 - 1.0   # normalize to [-1, 1]
    AB = lab[:, :, 1:]  / 128.0          # normalize to [-1, 1]
    return L, AB


def make_dataset(image_dir, batch_size=BATCH_SIZE):
    paths = tf.data.Dataset.list_files(os.path.join(image_dir, "*.jpg"), shuffle=True)

    def tf_preprocess(path):
        L, AB = tf.py_function(load_and_preprocess, [path], [tf.float32, tf.float32])
        L.set_shape([IMG_SIZE, IMG_SIZE, 1])
        AB.set_shape([IMG_SIZE, IMG_SIZE, 2])
        return L, AB

    return (
        paths
        .map(tf_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )


dataset = make_dataset(IMAGE_DIR)

num_images = len(tf.io.gfile.glob(os.path.join(IMAGE_DIR, "*.jpg")))
print(f"{num_images} images found")
print(f"{num_images // BATCH_SIZE} steps per epoch")

## Cell 5 – Generator (U-Net)

In [ ]:
def conv_block(x, filters, size=4, strides=2, use_bn=True):
    x = tf.keras.layers.Conv2D(filters, size, strides=strides, padding='same', use_bias=False)(x)
    if use_bn:
        x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    return x


def deconv_block(x, skip, filters, dropout=False):
    x = tf.keras.layers.Conv2DTranspose(filters, 4, strides=2, padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    if dropout:
        x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Concatenate()([x, skip])
    return x


def build_generator():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="L_input")

    e1 = conv_block(inputs, 64,  use_bn=False)  # 128x128
    e2 = conv_block(e1,     128)                 # 64x64
    e3 = conv_block(e2,     256)                 # 32x32
    e4 = conv_block(e3,     512)                 # 16x16
    e5 = conv_block(e4,     512)                 # 8x8
    e6 = conv_block(e5,     512)                 # 4x4
    b  = conv_block(e6,     512)                 # 2x2 (bottleneck)

    d1 = deconv_block(b,  e6, 512, dropout=True)  # 4x4
    d2 = deconv_block(d1, e5, 512, dropout=True)  # 8x8
    d3 = deconv_block(d2, e4, 512, dropout=True)  # 16x16
    d4 = deconv_block(d3, e3, 256)                # 32x32
    d5 = deconv_block(d4, e2, 128)                # 64x64
    d6 = deconv_block(d5, e1, 64)                 # 128x128

    outputs = tf.keras.layers.Conv2DTranspose(
        2, 4, strides=2, padding='same', activation='tanh', name="AB_output"
    )(d6)  # 256x256x2

    return tf.keras.Model(inputs, outputs, name="Generator")


generator = build_generator()
generator.summary()

## Cell 6 – Discriminator (PatchGAN)

In [ ]:
def build_discriminator():
    L  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="L_input")
    AB = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 2), name="AB_input")

    x = tf.keras.layers.Concatenate()([L, AB])
    x = conv_block(x, 64,  use_bn=False)  # 128x128
    x = conv_block(x, 128)                # 64x64
    x = conv_block(x, 256)                # 32x32

    x = tf.keras.layers.Conv2D(512, 4, strides=1, padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Conv2D(1, 4, strides=1, padding='same', name="patch_output")(x)

    return tf.keras.Model([L, AB], x, name="Discriminator")


discriminator = build_discriminator()
discriminator.summary()

## Cell 7 – Loss Functions & Optimizers

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    return bce(tf.ones_like(real_output), real_output) + bce(tf.zeros_like(fake_output), fake_output)


def generator_loss(fake_output, pred_AB, real_AB):
    gan_loss = bce(tf.ones_like(fake_output), fake_output)
    l1_loss  = tf.reduce_mean(tf.abs(real_AB - pred_AB))
    return gan_loss + LAMBDA * l1_loss, gan_loss, l1_loss


gen_optimizer  = tf.keras.optimizers.Adam(LR, beta_1=BETA_1)
disc_optimizer = tf.keras.optimizers.Adam(LR, beta_1=BETA_1)

print("Loss functions and optimizers ready")

## Cell 8 – Load Checkpoint (optional)

In [ ]:
LOAD_EPOCH = None   # e.g. 50, or None to start fresh

START_EPOCH = 0

if LOAD_EPOCH is not None:
    gen_path  = os.path.join(CHECKPOINT_DIR, f"gen_epoch_{LOAD_EPOCH:03d}.keras")
    disc_path = os.path.join(CHECKPOINT_DIR, f"disc_epoch_{LOAD_EPOCH:03d}.keras")

    if os.path.exists(gen_path) and os.path.exists(disc_path):
        generator     = tf.keras.models.load_model(gen_path)
        discriminator = tf.keras.models.load_model(disc_path)
        START_EPOCH   = LOAD_EPOCH
        print(f"Checkpoint epoch {LOAD_EPOCH} loaded — resuming training")
    else:
        print(f"Checkpoint not found at {gen_path} — starting from epoch 0")
else:
    print("No checkpoint — starting from epoch 0")

## Cell 9 – Training Step

In [ ]:
@tf.function
def train_step(L, real_AB):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_AB = generator(L, training=True)

        real_output = discriminator([L, real_AB], training=True)
        fake_output = discriminator([L, fake_AB], training=True)

        disc_loss                    = discriminator_loss(real_output, fake_output)
        gen_total, gan_loss, l1_loss = generator_loss(fake_output, fake_AB, real_AB)

    gen_grads  = gen_tape.gradient(gen_total,  generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(gen_grads,  generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_total, disc_loss, gan_loss, l1_loss


print("Training step compiled")

## Cell 10 – Visualization

In [ ]:
def visualize(epoch, n=4):
    L_batch, AB_batch = next(iter(dataset.take(1)))
    pred_AB = generator(L_batch[:n], training=False)

    fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
    fig.suptitle(f"Epoch {epoch}", fontsize=16, fontweight='bold')

    for i in range(n):
        L_np      = (L_batch[i].numpy()  + 1.0) * 50.0
        predAB_np =  pred_AB[i].numpy()  * 128.0
        realAB_np =  AB_batch[i].numpy() * 128.0

        pred_rgb = np.clip(color.lab2rgb(np.concatenate([L_np, predAB_np], axis=-1)), 0, 1)
        real_rgb = np.clip(color.lab2rgb(np.concatenate([L_np, realAB_np], axis=-1)), 0, 1)

        axes[i, 0].imshow(L_np[:, :, 0], cmap='gray')
        axes[i, 0].set_title("Input (B&W)")
        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title("GAN Colorized")
        axes[i, 2].imshow(real_rgb)
        axes[i, 2].set_title("Original")

        for ax in axes[i]:
            ax.axis('off')

    plt.tight_layout()
    save_path = os.path.join(PREVIEW_DIR, f"preview_epoch_{epoch:03d}.png")
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"  Preview saved: {save_path}")


print("Visualization function ready")

## Cell 11 – Training Loop

In [ ]:
history = {'gen_loss': [], 'disc_loss': [], 'gan_loss': [], 'l1_loss': []}

print(f"Training starts at epoch {START_EPOCH + 1}\n")

for epoch in range(START_EPOCH, EPOCHS):
    print(f"── Epoch {epoch + 1:3d}/{EPOCHS} " + "─" * 40)

    epoch_gen, epoch_disc, epoch_gan, epoch_l1 = [], [], [], []

    for step, (L, AB) in enumerate(dataset):
        g_loss, d_loss, gan_l, l1_l = train_step(L, AB)

        epoch_gen.append(float(g_loss))
        epoch_disc.append(float(d_loss))
        epoch_gan.append(float(gan_l))
        epoch_l1.append(float(l1_l))

        if step % 50 == 0:
            print(f"  Step {step:4d} | G: {g_loss:.4f} (GAN: {gan_l:.4f}, L1: {l1_l:.4f}) | D: {d_loss:.4f}")

    history['gen_loss'].append(np.mean(epoch_gen))
    history['disc_loss'].append(np.mean(epoch_disc))
    history['gan_loss'].append(np.mean(epoch_gan))
    history['l1_loss'].append(np.mean(epoch_l1))

    print(f"  Avg G-Loss: {history['gen_loss'][-1]:.4f} | Avg D-Loss: {history['disc_loss'][-1]:.4f}")

    if (epoch + 1) % SAVE_EVERY == 0:
        generator.save(os.path.join(CHECKPOINT_DIR, f"gen_epoch_{epoch+1:03d}.keras"))
        discriminator.save(os.path.join(CHECKPOINT_DIR, f"disc_epoch_{epoch+1:03d}.keras"))
        print(f"  Checkpoint saved (epoch {epoch + 1})")

    if (epoch + 1) % 5 == 0:
        visualize(epoch + 1)

print("\nTraining complete!")

## Cell 12 – Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Loss", fontsize=14, fontweight='bold')

axes[0].plot(history['gen_loss'],  label='Generator (total)', color='blue')
axes[0].plot(history['disc_loss'], label='Discriminator',     color='red')
axes[0].set_title("GAN Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['l1_loss'],  label='L1 Loss',  color='green')
axes[1].plot(history['gan_loss'], label='GAN Loss', color='orange')
axes[1].set_title("Generator Loss Breakdown")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "loss_curves.png"), dpi=100)
plt.show()

## Cell 13 – Test a Single Image

In [ ]:
def colorize_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0

    lab = color.rgb2lab(img.numpy()).astype(np.float32)
    L   = (lab[:, :, 0:1] / 50.0 - 1.0)[np.newaxis, ...]

    pred_AB = generator(L, training=False)[0].numpy() * 128.0
    L_np    = (L[0] + 1.0) * 50.0

    result_rgb = np.clip(color.lab2rgb(np.concatenate([L_np, pred_AB], axis=-1)), 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(L_np[:, :, 0], cmap='gray')
    axes[0].set_title("Input (B&W)")
    axes[1].imshow(result_rgb)
    axes[1].set_title("GAN Colorized")
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    return result_rgb


# Example usage:
# result = colorize_image(r"C:\path\to\your\image.jpg")